# Entrenamiento del modelo de traducción con el dataset creado durante la práctica 3

Tomamos el dataset de contratos en inglés y español creado durante la práctica 3 para entrenar un modelo de traducción.
https://huggingface.co/datasets/LourdesY/agreements_EN_ES

## Instalando librerías

Por defecto, el entorno de Google Colab no tiene instaladas las librerías de [HuggingFace](https://huggingface.co/), por lo que vamos a hacer en primer lugar es instalar las librerías: [Transformers](https://huggingface.co/docs/transformers/index), [Datasets](https://huggingface.co/docs/datasets/index), y [Evaluate](https://huggingface.co/docs/evaluate/index).

In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system ==

A continuación nos conectamos al hub de huggingface, lo que nos permitirá subir nuestros modelos a este entorno. Al ejecutar la siguiente celda aparecerá un widget en el cual tendremos que copiar el token generado en el primer paso y pulsar en el botón login.

In [2]:
from huggingface_hub import notebook_login

notebook_login()

## Dataset

Comenzamos descargando el dataset bilingüe de contratos que he creado con LF Aligner https://huggingface.co/datasets/LourdesY/agreements_EN_ES

In [3]:
from datasets import load_dataset

In [4]:
dataset = load_dataset("LourdesY/agreements_EN_ES")

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/13.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/70 [00:00<?, ? examples/s]

Veamos el contenido de este dataset.

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['EN', 'ES  '],
        num_rows: 70
    })
})

Observamos que el dataset solo tiene datos de entrenamiento, por lo que debemos proceder a dividirlo para obtener conjuntos de entrenamiento y prueba.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from datasets import Dataset

# Convertir a DataFrame para facilitar el manejo
df = pd.DataFrame(dataset['train'])

# Crear índices para la división
indices = np.arange(len(df))
train_indices, test_indices = train_test_split(indices, test_size=0.2, random_state=42)

# Crear una nueva columna 'split' para identificar a qué conjunto pertenece cada ejemplo
df['split'] = 'train'  # Por defecto, asignamos 'train'
df.loc[test_indices, 'split'] = 'test'  # Actualizamos los índices de test

print(f"Tamaño del dataset: {len(df)}")
print(f"Ejemplos en training: {len(df[df['split'] == 'train'])}")
print(f"Ejemplos en test: {len(df[df['split'] == 'test'])}")

#dataset dividido
dataset_split = dataset["train"].train_test_split()

# Si necesitas filtrar el dataset por split
train_examples = df[df['split'] == 'train']
test_examples = df[df['split'] == 'test']

Tamaño del dataset: 70
Ejemplos en training: 56
Ejemplos en test: 14


Podemos ver que tenemos un objeto DatasetDict que puede verse como un diccionario. Dicho diccionario contiene un atributo `train` y un atributo `test`, por lo que en este caso ya disponemos de la división para entrenar nuestro modelo. Vamos a ver alguna de las frases del dataset, para lo que tenemos que acceder al atributo `train`.

In [7]:
dataset_split


DatasetDict({
    train: Dataset({
        features: ['EN', 'ES  '],
        num_rows: 52
    })
    test: Dataset({
        features: ['EN', 'ES  '],
        num_rows: 18
    })
})

Con el anterior comando vemos que tenemos un Dataset con dos columnas: `EN`, y `ES`. Si queremos ver el contenido del dataset, lo podemos transformar a formato pandas y verlo como una tabla. Vemos a continuación que cada fila de nuestro dataset contiene una frase en inglés y otra frase alineada que está traducida al español.

In [8]:
dataset_split['train'].to_pandas()

,EN,ES
0,10.4 This Consulting Agreement shall be govern...,10.4 El presente Contrato de Consultoría se re...
1,"THIS AGREEMENT made the 28th day of August, 2023",EL PRESENTE CONTRATO se celebra el día 28 de a...
2,"As an independent contractor, the Consultant s...","Como contratista independiente, el Consultor s..."
3,3.1 The Consultant shall perform the Consultin...,3.1 El Consultor deberá prestar los Servicios ...
4,9. Non-compete and Non-circumvent,9. Prohibición de competencia y circunvención
5,"7.2 The Consultant will not be eligible for, a...",7.2 El Consultor no tendrá derecho a ni partic...
6,6.1 The Consultant shall be entitled to take s...,6.1 El Consultor tendrá derecho a tomarse las ...
7,10. General,10. General
8,"Such information may include, without limitati...","Dicha información puede incluir, sin carácter ..."
9,ContractorY (the “Consultant”),EmployeeY (el «Consultor»)


Para poder entrenar un modelo con este dataset es necesario tokenizarlo. Cada modelo tokeniza de una manera distinta, por lo que es necesario indicar el modelo para tokenizar el texto. En nuestro caso vamos a utilizar el modelo [mbart](https://huggingface.co/facebook/mbart-large-50).

In [9]:
from transformers import AutoTokenizer

model_checkpoint = "facebook/mbart-large-50"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

Definimos una función para tokenizar el texto.

In [17]:
max_length = 128


def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["EN"],
        max_length=max_length,
        truncation=True,
    )
    labels = tokenizer(
        examples["ES  "],
        max_length=max_length,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

Tokenizamos el dataset y lo mostramos.

In [18]:
dftrain = dataset_split["train"].to_pandas()
dftrain

,EN,ES
0,10.4 This Consulting Agreement shall be govern...,10.4 El presente Contrato de Consultoría se re...
1,"THIS AGREEMENT made the 28th day of August, 2023",EL PRESENTE CONTRATO se celebra el día 28 de a...
2,"As an independent contractor, the Consultant s...","Como contratista independiente, el Consultor s..."
3,3.1 The Consultant shall perform the Consultin...,3.1 El Consultor deberá prestar los Servicios ...
4,9. Non-compete and Non-circumvent,9. Prohibición de competencia y circunvención
5,"7.2 The Consultant will not be eligible for, a...",7.2 El Consultor no tendrá derecho a ni partic...
6,6.1 The Consultant shall be entitled to take s...,6.1 El Consultor tendrá derecho a tomarse las ...
7,10. General,10. General
8,"Such information may include, without limitati...","Dicha información puede incluir, sin carácter ..."
9,ContractorY (the “Consultant”),EmployeeY (el «Consultor»)


In [19]:
dftrain = dftrain.dropna()

In [20]:
from datasets import DatasetDict
dataset_split = DatasetDict({"train":Dataset.from_pandas(dftrain),"test":dataset_split["test"]})

In [22]:
tokenized_datasets = dataset_split.map(preprocess_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['EN', 'ES  ', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 52
    })
    test: Dataset({
        features: ['EN', 'ES  ', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 18
    })
})

Al igual que ocurría con los modelos de clasificación de texto, podemos ver que han aparecido tres nuevas columnas ('input_ids', 'labels' y 'attention_mask') que serán utilizadas para entrenar el modelo. Notar que ya disponemos de una columna labels que será la que tendrá que predecir el modelo.  

Por último, antes de definir nuestro modelo tenemos que definir una función que se va a encargar de preparar los datos para que sean procesados de manera eficiente por el modelo.

In [23]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer)

## Modelo

Pasamos ahora a definir el modelo, lo primero que vamos a definir son los argumentos con los que vamos a entrenar nuestro modelo.

In [24]:
from transformers import Seq2SeqTrainingArguments


args = Seq2SeqTrainingArguments(
    # Carpeta donde se guardará el modelo
    output_dir="mbart-contratos-en-es",
    # Evaluar el modelo cada época
    evaluation_strategy="epoch",
    # Valores para modificar los pesos del modelo
    learning_rate=5.6e-5,
    weight_decay=0.01,
    # Número de ejemplos que se muestran cada vez al modelo
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    # Número de veces que se muestran todos los datos al modelo
    num_train_epochs=2,
    # Realizar predicciones al validar
    predict_with_generate=True,
    push_to_hub=True,
    report_to="none"
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


A continuación definimos nuestro modelo, para ello usamos la clase `AutoModelForSeq2SeqLM` y vamos a utilizar un modelo pre-entrenado (recordar lo que era el transfer learning). Para ello solo tenemos que indicar el nombre de nuestro modelo (definido previamente en la variable `model_checkpoint`.  

In [25]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Ahora definimos la función que usaremos para calcular la precisión de nuestro modelo. En este caso usaremos la métrica [sacrebleu](https://huggingface.co/spaces/evaluate-metric/sacrebleu) que nos proporciona la métrica bleu utilizada en el Workshop on Machine Translation (WMT).

In [26]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.8 MB/s eta 0:00:00


In [27]:
!pip install sacrebleu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.7 MB/s eta 0:00:00


In [28]:
import numpy as np

import evaluate

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


Ya podemos definir nuestro objeto `trainer` que usaremos para entrenar nuestro modelo (en este caso será un modelo secuencia a secuencia). La estructura de este objeto será siempre la misma. Le tenemos que proporcionar:
1. El modelo.
2. La configuración del entrenamiento.
3. El conjunto de entrenamiento.
4. El conjunto de test.
5. El objeto que prepara los datos.
6. El tokenizador.
7. La métrica.

In [29]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

<ipython-input-29-7e05dcb5bafc>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Y ahora entrenamos el modelo mediante el método `train`.

In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,No log,3.158582,18.585900,29.722200
2,No log,2.999468,23.052800,31.722200


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3353: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=14, training_loss=3.7838262830461775, metrics={'train_runtime': 108.6086, 'train_samples_per_second': 0.958, 'train_steps_per_second': 0.129, 'total_flos': 17243919286272.0, 'train_loss': 3.7838262830461775, 'epoch': 2.0})

## Compartiendo el modelo

Una vez que tenemos entrenado nuestro modelo, nos interesa compartirlo con el resto del mundo para que puedan usarlo y también compararlo con otros modelos.

Es por ello que vamos a subir nuestro modelo al hub de huggingface. Para ello tenemos que ejecutar el siguiente comando.

In [31]:
%cd mbart-contratos-en-es
trainer.push_to_hub(commit_message="Training complete", tags="simplification")

/content/mbart-contratos-en-es


CommitInfo(commit_url='https://huggingface.co/LourdesY/mbart-contratos-en-es/commit/4116e9e0828226e50a856e14ab9cdda6d196c012', commit_message='Training complete', commit_description='', oid='4116e9e0828226e50a856e14ab9cdda6d196c012', pr_url=None, repo_url=RepoUrl('https://huggingface.co/LourdesY/mbart-contratos-en-es', endpoint='https://huggingface.co', repo_type='model', repo_id='LourdesY/mbart-contratos-en-es'), pr_revision=None, pr_num=None)

El modelo está disponible en https://huggingface.co/LourdesY/mbart-contratos-en-es.

Finalmente, vamos a probar cómo funciona el modelo.


## Usando el modelo

En este caso al ser un modelo que hemos entrenado nosotros mismos podríamos usar los ficheros locales, pero vamos a ver cómo usar el modelo que acabamos de subir al hub de HuggingFace.

Para ello usamos un `pipeline` al que le debemos indicar el nombre del modelo que queremos descargar.

In [32]:
from transformers import pipeline
en_es = pipeline('text2text-generation', model='LourdesY/mbart-contratos-en-es')

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/226 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/992 [00:00<?, ?B/s]

Device set to use cuda:0


Ahora podemos hacer traducciones con nuestro modelo.


Procedo a probar con oraciones que suelen darse en contratos, procurando que no estén formuladas exactamente como en el dataset de entrenamiento.

Los dos primeros ejemplos están bien en tanto que transmiten el mensaje del original. La formulación no es del todo correcta, pero es normal dado el reducido tamaño del dataset de entrenamiento.

In [38]:
en_es('This Consulting Agreement is entered into as of 27.03.2025')

[{'generated_text': 'Este Contrato de Consultoría es celebrado el día 27.03.2025'}]

In [39]:
en_es('The Company desires to retain Consultant as an independent contractor to perform consulting services for the Company')

[{'generated_text': 'La Empresa desea contratar a un Consultor independiente para prestar los servicios de consultoría para la Empresa.'}]

A continuación, he querido probar con estas tres frases para poner a prueba el modelo.
El término "term" en contratos tiene una doble acepción. Por un lado se refiere al plazo o duración en la que el contrato está vigente (primera frase) y, por otro, se puede referir a los términos y condiciones del contrato.
Pese a que el dataset contiene un párrafo en el que term se traduce con esta acepción de plazo o duración del contrato, el modelo no la ha captado y ha utilizado "término".

Es curioso, sin embargo, que en la tercera frase (donde "term" sí es "término" o "disposición") no haya elegido traducirlo por "término", sino otra palabra como "parte" (también correcta).

La segunda frase contiene un falso amigo "indemnify and hold harmless", que no siempre se refiere a indemnizar a alguien, sino a eximir de responsabilidad a la otra parte ante cualquiera de los supuestos que se indiquen en el contrato. A pesar de que el dataset contenía esta expresión y su correcta traducción, no ha podido trasladarlo. Seguramente con más datos de entrenamiento, esto se solucionaría.

In [46]:
en_es(['The term of this Agreement will begin on the Effective Date of this Agreement',
       'Consultant agrees to indemnify and hold harmless the Company',
       'If any term of this Agreement is not valid, the rest of this Agreement will remain in effect'])

[{'generated_text': 'El término del presente Contrato se extenderá a partir del día siguiente al de entrada en vigor del presente.'},
 {'generated_text': 'Consultor se compromete a indemnizar y mantener indemne a la Empresa'},
 {'generated_text': 'Si ninguna de las partes del presente Contrato es válida, el resto del presente Contrato permanecerá en vigor.'}]

Finalmente, si probamos con un fragmento algo más extenso, el modelo falla y no traduce todo el contenido.
El fragmento de prueba es:

*Termination. This Agreement will terminate upon the entry of the Parties to a full-time employment agreement. Otherwise, either Party may terminate this Agreement at any time, with or without cause, upon thirty (30) days’ notice, but, if (and only if) such termination is without cause, Company shall upon such termination pay Consultant all unpaid, undisputed amounts due for the Services completed prior to notice of such termination.*

Y la traducción resultante es

*Terminación. El presente Contrato de Consultoría se entenderá terminado en virtud de la entrada en vigor del presente Contrato de Consultoría.*

No solo no ha traducido íntegramente el contenido, sino que lo poco que se ha traducido no tiene sentido.

In [44]:
en_es("Termination. This Agreement will terminate upon the entry of the Parties to a full-time employment agreement. Otherwise, either Party may terminate this Agreement at any time, with or without cause, upon thirty (30) days’ notice, but, if (and only if) such termination is without cause, Company shall upon such termination pay Consultant all unpaid, undisputed amounts due for the Services completed prior to notice of such termination.")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[{'generated_text': 'Terminación. El presente Contrato de Consultoría se entenderá terminado en virtud de la entrada en vigor del presente Contrato de Consultoría.'}]

Por curiosidad, voy a probar con frases algo más alejadas del contenido del dataset de entrenamiento, para ver qué sucede.

Me sorprende que haya podido realizar las traducciones, ya que el dataset no contiene traducciones para birthday, book o holidays, por ejemplo. No obstante, destaca el hecho de que haya traducido "cousin" por "padre".

In [36]:
en_es(["Today it's my birthday",
       "My cousin has bought a book",
       "Next week we'll go on holidays"])


[{'generated_text': 'Hoy es mi día de cumpleaños'},
 {'generated_text': 'Mi padre ha comprado un libro'},
 {'generated_text': 'El próximo fin de semana nos vamos de vacaciones'}]